# Keras 3 + MLflow — control hub

**Read [`slides/part1.html`](../../slides/part1.html) first.** The deck explains the words this
notebook uses without stopping to define them: container, image, layer, volume, registry, alias,
endpoint. This notebook only drives the work.

Every cell below gives orders to three **containers**. Nothing trains in this notebook's own
Python process — its **kernel**.

| container | role | lifecycle |
|---|---|---|
| `mlflow-server` | tracking + model registry, sqlite backend | long running, port 5000 |
| `trainer` | Keras 3 (torch backend), one run per launch | launched per run, exits |
| `api-serve` | FastAPI + uvicorn, serves `@champion` | you start it, port 8000 |

The flow: **train a few runs → compare them → promote one to `@champion` → serve it → watch the
predictions**.

Each section is tagged `A1`, `B3`, `C2`, and so on. The next cell shows what those codes mean.
Part 2 (`part2/notebooks/hub.ipynb`) uses the same scheme, so the two notebooks read as one
course.

In [ ]:
import json, os, time
from pathlib import Path

# Walk up to the repo root (the folder holding docker-compose.yml). The notebook
# lives in part1/notebooks, so that is two levels up -- but searching rather than
# counting keeps this working wherever the notebook is opened from.
PROJECT = next(d for d in [Path.cwd(), *Path.cwd().parents]
               if (d / "docker-compose.yml").exists())
os.chdir(PROJECT)     # every `!docker ...` line below runs from the project root
print("project:", PROJECT)

# .env is what merges compose.part1.yml into every `docker compose` call, and what sets
# DOCKER_UID/DOCKER_GID. Without it the base file loads alone and compose reports
# "no such service: trainer" instead of "you forgot .env".
print(".env   :", "found" if (PROJECT / ".env").exists() else "MISSING -> cp .env.example .env")

!docker compose version

## The map

Five stages, one registry in the middle, and **one loop that you close by hand**. Every section
below is tagged with its code from this diagram, and so is the file it drives: `B3` is
`part1/trainer/train.py`, and `D1` is `part1/api-serve/app.py`.

![Part 1 architecture: stages A to E around the MLflow registry, and one manual loop](architecture.svg)

Slide 1 of the deck builds this same picture one stage at a time. Two sentences from it are worth
keeping in mind while you work. The three containers **share nothing but the registry**, so
promoting a different version changes what is served without touching a line of code. And the
thing that closes the loop is **you**.

---
# A · Data

### A1 · Get the digits

The trainer container downloads MNIST on its first run and saves it to `./data/mnist.npz`, so the
file is usually on disk already. If it is not, the next cell downloads the same file directly. It
is the exact archive `keras.datasets.mnist` uses, and downloading it here keeps Keras out of the
host virtual environment.

In [ ]:
import urllib.request

import numpy as np

MNIST = PROJECT / "data" / "mnist.npz"

if not MNIST.exists():
    # Same archive keras.datasets.mnist pulls, ~11 MB. Written where the trainer expects its
    # cache, so this download also saves the container the trouble later.
    MNIST.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", MNIST)
    print("downloaded", MNIST)

d = np.load(MNIST)
x_train, y_train, x_test, y_test = d["x_train"], d["y_train"], d["x_test"], d["y_test"]

print(f"train {x_train.shape}  test {x_test.shape}  dtype {x_train.dtype}  "
      f"pixels {x_train.min()}-{x_train.max()}")
print(f"classes {sorted(set(y_train.tolist()))}   "
      f"smallest class {np.bincount(y_train).min():,}   largest {np.bincount(y_train).max():,}")

In [ ]:
import matplotlib.pyplot as plt

# Palette shared with Part 2's charts and both diagrams: one categorical hue for data,
# greys for ink and grid. Digits are images, so they stay greyscale.
BLUE, INK, SUB, MUTED, GRID, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "axes.labelcolor": SUB, "text.color": SUB, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8,
})

fig = plt.figure(figsize=(11, 6.4))
gs = fig.add_gridspec(3, 10, height_ratios=[1.15, 1.15, 1.5], hspace=0.4, wspace=0.15)
fig.suptitle("A2 · What is actually in this dataset", x=0.055, ha="left",
             fontsize=13, fontweight="bold", color=INK)

rng = np.random.default_rng(0)
for c in range(10):                       # one real example per class
    ax = fig.add_subplot(gs[0, c])
    ax.imshow(x_train[rng.choice(np.flatnonzero(y_train == c))], cmap="gray_r", vmin=0, vmax=255)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_color(GRID)
    if c == 0:
        ax.set_ylabel("one\nexample", rotation=0, ha="right", va="center", fontsize=8, color=SUB)

for c in range(10):                       # every example of that class, averaged
    ax = fig.add_subplot(gs[1, c])
    ax.imshow(x_train[y_train == c].mean(0), cmap="gray_r", vmin=0, vmax=255)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_color(GRID)
    ax.set_xlabel(str(c), fontsize=9, color=SUB, labelpad=2)
    if c == 0:
        ax.set_ylabel("all 6k,\naveraged", rotation=0, ha="right", va="center", fontsize=8, color=SUB)

ax = fig.add_subplot(gs[2, :])
counts = np.bincount(y_train)
ax.bar(range(10), counts, width=0.7, color=BLUE)
for c, n in enumerate(counts):
    ax.text(c, n + 90, f"{n:,}", ha="center", fontsize=8, color=SUB)
ax.set_xticks(range(10)); ax.set_ylim(0, counts.max() * 1.18)
ax.set_xlabel("digit"); ax.set_ylabel("training images")
ax.set_title("Every class is equally common — about 6,000 each")
ax.grid(axis="y"); ax.set_axisbelow(True);

Look at your own picture before you move on. Slide 9 of the deck says why this dataset was
chosen: the classes are balanced, so accuracy is an honest score here; the average of each class
still looks like the digit, so almost no labels are wrong; and a few digits are genuinely
ambiguous, which is why a good score is 99-point-something and not 100.

None of this is MLOps. It is the ten minutes of looking at data that belongs before any of it.
Part 2 opens with the same section, on a dataset that hides much more.

---
# B · Train

Three containers first, then three models. Again, nothing trains in this notebook's kernel.

### B1 · Build the images

The first build downloads CPU-only torch, so expect it to take a few minutes. Later builds reuse
the layers that have already been built and are fast — slide 6.

In [4]:
!docker compose --profile tools --profile serve build

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.1s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.59kB                                           0.0s
[+] Building 0.3s (4/5)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.59kB                                           0.0s
 => [trainer internal] load build definition from Dockerfile               0.0s
 => => transferring dockerfile: 393B                                       0.0s
 => [api-serve internal] load build definition from Dockerfile             0.0s
 => => transferring dockerfile: 370B                                       0.0s
 => [mlflow-server internal] load bui

### B2 · Start (or restart) the MLflow server

The tracking server is reusable and standalone: it knows nothing about Keras. Open
<http://localhost:5000> once it reports itself healthy.

The cell forces the container to be created again, so **re-run it whenever you want a clean
restart** — after changing `docker-compose.yml`, or when the server stops responding. The tracking
data lives in `./mlruns` on the host, so a restart never loses runs, models, or aliases.

In [ ]:
import urllib.request

# --force-recreate always tears the container down and builds a fresh one, so
# re-running this cell is a real restart instead of a no-op "Container Running".
# Safe to repeat: all state lives in ./mlruns on the host, not in the container.
!docker compose up -d --force-recreate mlflow-server

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:5000/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    # Same as D1: the health timeout says nothing about the cause, and the cause is
    # always in the container log. Most often it is "unable to open database file" --
    # ./mlruns owned by root, so the container's ${DOCKER_UID} cannot create mlflow.db.
    !docker compose logs --tail 30 mlflow-server
    raise RuntimeError("mlflow-server did not come up — see the log above")

print("UI: http://localhost:5000")

### B3 · Train — three runs, one cell at a time

Each cell launches the **trainer container** and passes its hyperparameters as `-e VAR=value`
(slide 7). The container trains one model, records one MLflow run, registers a model version, and
then exits. No rebuild: changing a number and re-running the cell is the whole edit cycle.

Three runs are enough to make the comparison in the MLflow UI mean something: a baseline, a better
architecture, and one run you expect to lose.

In [6]:
def train(name, **params):
    """Launch the trainer container for ONE run.

    Every keyword becomes an -e VAR=value on `docker compose run`, so the whole
    command is visible in the cell output. Knobs: kind (mlp/cnn), size, dropout,
    opt (adam/sgd), lr, epochs, batch_size, train_subset.
    """
    env = " ".join(f"-e {k.upper()}={v}"
                   for k, v in dict(run_name=name, **params).items())
    !docker compose --profile tools run --rm {env} trainer


print(train.__doc__.splitlines()[0])

Launch the trainer container for ONE run.


#### Run 1 — the baseline

One hidden layer of 128 units. Note the accuracy and the number of seconds. The other two runs are
judged against these two numbers.

In [7]:
train("mlp-128", kind="mlp", size=128, dropout=0.2, opt="adam", lr="1e-3")

[+] create 1/1
 ✔ Container mlflow-server Running                                          0.0s
[+] start 1/1
 ✔ Container mlflow-server Running                                          0.0s
[+]  1/1
 ✔ Container mlflow-server Running                                          0.0s
Container mlflow-server Waiting 
Container mlflow-server Healthy 
Container keras-mlflow-mnist-trainer-run-578ba9a2da48 Creating 
Container keras-mlflow-mnist-trainer-run-578ba9a2da48 Created 
[keras] 3.15.1 backend=torch  mlflow=3.15.1
[config] {'name': 'mlp-128', 'kind': 'mlp', 'size': 128, 'dropout': 0.2, 'opt': 'adam', 'lr': 0.001} epochs=3 batch_size=128 subset=20000
[data] loaded /data/mnist.npz
[data] train=(20000, 28, 28, 1) test=(10000, 28, 28, 1)
Epoch 1/3
141/141 - 1s - 6ms/step - accuracy: 0.8101 - loss: 0.6682 - val_accuracy: 0.9275 - val_loss: 0.2798
Epoch 2/3
141/141 - 1s - 6ms/step - accuracy: 0.9116 - loss: 0.3115 - val_accuracy: 0.9425 - val_loss: 0.2225
Epoch 3/3
141/141 - 1s - 7ms/step - ac

#### Run 2 — a different architecture

A convolutional block instead of a dense layer, and 5 epochs instead of the default 3. It has
fewer parameters than run 1 and a clearly better score. This is the run that becomes `@champion`.

In [ ]:
train("cnn-32", kind="cnn", size=32, dropout=0.25, opt="adam", lr="1e-3", epochs=5)

#### Run 3 — a deliberately bad control

No dropout, and SGD with a learning rate that is too large. A comparison needs at least one run
you expect to lose. Without one, you cannot tell a good model from an easy dataset.

In [ ]:
train("cnn-16-sgd", kind="cnn", size=16, dropout=0.0, opt="sgd", lr="3e-2")

---
# C · Select

### C1–C2 · Compare the runs, then promote the winner to `@champion`

**C1.** Compare the three runs in the MLflow UI: open <http://localhost:5000>, choose the
experiment `mnist-keras`, and sort by `test_accuracy`. You can also tick two runs and press
**Compare**.

**C2.** Then promote one of them by moving the `@champion` alias — one API call plus a restart,
as on slide 4. The cell picks the best run by `test_accuracy`. To promote a different one, pass
its name as `RUN_NAME`.

In [ ]:
import mlflow
from mlflow import MlflowClient

RUN_NAME = None      # None = whichever run scored best; or e.g. "cnn-32"

mlflow.set_tracking_uri("http://localhost:5000")
runs = mlflow.search_runs(
    experiment_names=["mnist-keras"],
    filter_string=f"tags.mlflow.runName = '{RUN_NAME}'" if RUN_NAME else "",
    order_by=["metrics.test_accuracy DESC"], max_results=1)

if runs.empty:
    raise RuntimeError("no matching run — train at least one model first")

best = runs.iloc[0]
client = MlflowClient()

# find the registered version produced by that run
version = next(v for v in client.search_model_versions("name='mnist-keras'")
               if v.run_id == best["run_id"])

client.set_registered_model_alias("mnist-keras", "champion", version.version)
client.set_model_version_tag("mnist-keras", version.version, "promoted_by", "hub-notebook")

print(f"champion -> version {version.version} "
      f"({best['tags.mlflow.runName']}, acc={best['metrics.test_accuracy']:.4f})")

---
# D · Serve

### D1 · Start the serving container

The server resolves the alias once, at startup. Re-run this cell after promoting a different
version, or send `POST /reload` to switch version without a restart.

In [ ]:
import requests

!docker compose --profile serve up -d --force-recreate api-serve

for _ in range(60):
    try:
        health = requests.get("http://localhost:8000/health", timeout=2).json()
        break
    except Exception:
        time.sleep(2)
else:
    !docker compose logs --tail 50 api-serve
    raise RuntimeError("api-serve did not come up")

print(json.dumps(health, indent=2))

### D2 · Example inference

Two ways in, and both reach the same champion:

- `GET /predict/sample/{i}` — the container looks the digit up in its own cached copy of the
  MNIST test set. Convenient for `curl`. The cell asks for the first ten.
- `POST /predict` — you send 784 raw pixel values. The cell draws three of those ten digits,
  chosen at random, and posts them, so you can compare the prediction against what you can see.

Both indexes refer to the same test set, so `sample 3` and the third drawing are the same image.

```bash
curl localhost:8000/health
curl localhost:8000/predict/sample/7
```

In [17]:
import numpy as np

# a) ten digits, picked by index — the container reads them from its own cache
for i in range(10):
    r = requests.get(f"http://localhost:8000/predict/sample/{i}").json()
    hit = "  " if r["prediction"] == r["label"] else "<-- miss"
    print(f"sample {i}: true={r['label']} pred={r['prediction']} "
          f"conf={r['confidence']} {hit}")

# b) three of those ten at random, sent as raw pixels straight from the npz
d = np.load(PROJECT / "data" / "mnist.npz")
picks = sorted(np.random.choice(10, size=3, replace=False))

art = [["".join(" .:-=+*#%@"[p // 26] for p in row) for row in d["x_test"][i]]
       for i in picks]
print("\n" + "  ".join(f"sample {i}".ljust(28) for i in picks))
for line in zip(*art):
    print("  ".join(line))

for i in picks:
    img, truth = d["x_test"][i], int(d["y_test"][i])
    r = requests.post("http://localhost:8000/predict",
                      json={"pixels": img.flatten().tolist(), "label": truth}).json()
    print(f"sample {i}: true={truth}  pred={r['prediction']}  conf={r['confidence']}")

sample 0: true=7 pred=7 conf=1.0   
sample 1: true=2 pred=2 conf=0.9996   
sample 2: true=1 pred=1 conf=0.9977   
sample 3: true=0 pred=0 conf=1.0   
sample 4: true=4 pred=4 conf=0.9988   
sample 5: true=1 pred=1 conf=0.9981   
sample 6: true=4 pred=4 conf=0.9989   
sample 7: true=9 pred=9 conf=0.8961   
sample 8: true=5 pred=5 conf=0.9099   
sample 9: true=9 pred=9 conf=0.9938   

sample 1                      sample 4                      sample 6                    
                                                                                        
                                                                                        
                                                                                        
          ==*@@+-                                                                       
         *@@@@@@%.                                                                      
        *@@@%+*@@=                      .%       :.                   #+.        

---
# E · Observe

### E1 · Inference log

Every prediction was appended to `./inference/predictions.jsonl` on the host. You therefore never
have to open a shell inside the container to see what a running model is doing. Note the `version`
column: it names the champion you promoted in `C2`.

In [18]:
import pandas as pd

log = PROJECT / "inference" / "predictions.jsonl"
rows = [json.loads(l) for l in log.read_text().splitlines()]
pd.DataFrame(rows)[["ts", "model", "version", "source", "label", "prediction", "confidence"]].tail(10)

,ts,model,version,source,label,prediction,confidence
51,2026-08-17T10:26:45.916307+00:00,mnist-keras,20,mnist_test[3],0,0,1.0000
52,2026-08-17T10:26:45.958397+00:00,mnist-keras,20,mnist_test[4],4,4,0.9988
53,2026-08-17T10:26:46.002878+00:00,mnist-keras,20,mnist_test[5],1,1,0.9981
54,2026-08-17T10:26:46.047013+00:00,mnist-keras,20,mnist_test[6],4,4,0.9989
55,2026-08-17T10:26:46.091207+00:00,mnist-keras,20,mnist_test[7],9,9,0.8961
56,2026-08-17T10:26:46.133523+00:00,mnist-keras,20,mnist_test[8],5,5,0.9099
57,2026-08-17T10:26:46.175441+00:00,mnist-keras,20,mnist_test[9],9,9,0.9938
58,2026-08-17T10:26:46.255033+00:00,mnist-keras,20,request,2,2,0.9996
59,2026-08-17T10:26:46.283368+00:00,mnist-keras,20,request,4,4,0.9988
60,2026-08-17T10:26:46.310198+00:00,mnist-keras,20,request,4,4,0.9989


---
# Teardown

There is no cell here, on purpose. The notebook must never shut the stack down while you are still
using it, and a **Run All** would do exactly that. When you have finished exploring the MLflow UI,
run this in a terminal, from the project root:

```bash
docker compose --profile tools --profile serve down
```

Nothing is lost. Runs, models and the `@champion` alias live in `./mlruns` on the host and not
inside the containers, and `./inference` and `./data` survive as well. Restart from `B2`.

---

### Next

You closed loop ① by hand: you read the log, changed a number, trained, compared, and promoted.
That is the whole of MLOps in miniature.

The bottom band of the diagram names everything that is missing. **`part2/notebooks/hub.ipynb`**
adds all four:

- an application that has to survive this endpoint,
- humans supplying the labels the dataset stopped giving you,
- a dataset identified by a hash,
- and a scheduler that does the parts you just did with your hands.